# 02 — The zero-shot baseline, and why it is a floor

Before training anything, ask what the pretrained model already knows. The standard answer
for mutation effects is **masked-marginal scoring** (Meier et al. 2021, ESM-1v): mask the
mutated position, and score the mutation by how much more or less likely the model thinks
the mutant residue is than the wild-type one.

$$\text{score} = \log p(x_i = \text{mut} \mid x_{\setminus i}) - \log p(x_i = \text{wt} \mid x_{\setminus i})$$

No training, no labels. It costs one forward pass per mutation.

**It scores ≈0 on this task, and the interesting part is why.** This notebook runs the
scoring live on the 35M model, then reproduces the ablation that explains the failure.

In [1]:
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from scipy import stats
from transformers import AutoModelForMaskedLM, AutoTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

METRICS = json.loads((ROOT / "results/metrics.json").read_text())
complexes = pd.read_csv(ROOT / "data/processed/complexes.csv")
mutations = pd.read_csv(ROOT / "data/processed/mutations.csv")
print(f"{len(mutations):,} mutations across {complexes.shape[0]} complexes")

4,829 mutations across 316 complexes


## 1. Scoring a handful of mutations by hand

The 35M model is used here so the notebook runs on CPU in seconds. The reported baseline
uses 150M; both land in the same place.

Note `position + 1` when indexing the logits. ESM-2 prepends `<cls>`, so token *i+1* holds
residue *i*. Getting this wrong scores the neighbouring residue and produces a plausible,
completely meaningless number — which is why the assertion below is load-bearing rather
than decorative.

In [2]:
MODEL = "facebook/esm2_t12_35M_UR50D"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForMaskedLM.from_pretrained(MODEL).eval()

print(f"vocab {tok.vocab_size} | mask id {tok.mask_token_id} | "
      f"cls {tok.cls_token_id} | eos {tok.eos_token_id} | pad {tok.pad_token_id}")


@torch.no_grad()
def masked_marginal(sequence: str, position: int, wt_aa: str, mut_aa: str) -> float:
    """log p(mut) - log p(wt) at a masked position. `position` is 0-based into `sequence`."""
    # THE LOAD-BEARING ASSERT: an off-by-one here silently scores the wrong residue.
    assert sequence[position] == wt_aa, (
        f"expected {wt_aa} at {position}, found {sequence[position]}")

    enc = tok(sequence, return_tensors="pt")
    ids = enc["input_ids"].clone()
    ids[0, position + 1] = tok.mask_token_id          # +1 for the leading <cls>

    logits = model(input_ids=ids, attention_mask=enc["attention_mask"]).logits
    log_probs = torch.log_softmax(logits[0, position + 1], dim=-1)
    return float(log_probs[tok.convert_tokens_to_ids(mut_aa)]
                 - log_probs[tok.convert_tokens_to_ids(wt_aa)])

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

vocab 33 | mask id 32 | cls 0 | eos 2 | pad 1


In [3]:
# A small complex, so this stays quick on CPU.
seqs = dict(zip(complexes.pdb_field, complexes.sequence))
small = complexes.assign(n=complexes.sequence.str.len()).nsmallest(40, "n")
sample = (mutations[mutations.pdb_field.isin(small.pdb_field)]
          .sort_values("ddg", key=abs, ascending=False)
          .groupby("pdb_field").head(2).head(8))

rows = []
for r in sample.itertuples():
    score = masked_marginal(seqs[r.pdb_field], r.mutation_index, r.wt_aa, r.mut_aa)
    rows.append({"complex": r.pdb_field, "mutation": r.mutation,
                 "measured ΔΔG": round(r.ddg, 2), "masked-marginal": round(score, 2)})
scored = pd.DataFrame(rows)
scored

,complex,mutation,measured ΔΔG,masked-marginal
0,1BRS_A_D,HA100L,7.66,1.19
1,1BRS_A_D,DD39A,7.65,0.05
2,3EQS_A_B,WB7A,6.41,1.42
3,3EQY_A_C,WC7A,6.04,1.48
4,2GYK_A_B,AA49D,-5.91,0.30
5,1EMV_A_B,DA49A,5.91,-0.30
6,1B3S_A_D,AA102H,-5.68,-0.90
7,3EQY_A_C,FC3A,5.66,0.55


The sign convention is worth being explicit about. A **negative** masked-marginal means the
model prefers the wild-type residue, i.e. it thinks the mutation is disruptive; a
**positive** ΔΔG means the mutation *does* disrupt binding. So the expected relationship is
**negative** — and the baseline is scored with that in mind rather than by flipping signs
after seeing the result.

## 2. What it scores over the whole dataset

The full runs are recorded in `results/metrics.json`. Reproduce them with:

```
python -m src.baselines.zero_shot --backbone facebook/esm2_t30_150M_UR50D
python -m src.baselines.zero_shot --backbone facebook/esm2_t30_150M_UR50D --context chain
```

In [4]:
SPLITS = ["split_mutation", "split_pdb_id", "split_hold_out_proteins"]
rows = []
for key, label in [("zero_shot_esm2_t12_35M_UR50D", "ESM-2 35M"),
                   ("zero_shot_esm2_t30_150M_UR50D", "ESM-2 150M"),
                   ("zero_shot_esm2_t30_150M_UR50D_context-chain", "ESM-2 150M, mutated chain only")]:
    e = METRICS[key]["splits"]
    rows.append({"scoring": label,
                 **{s: round(e[s]["test"]["spearman"], 3) for s in SPLITS}})
pd.DataFrame(rows).set_index("scoring")

,split_mutation,split_pdb_id,split_hold_out_proteins
scoring,,,
ESM-2 35M,0.038,0.018,-0.004
ESM-2 150M,-0.082,0.005,-0.097
"ESM-2 150M, mutated chain only",-0.134,0.004,-0.161


**≈0 on every split.** That flatness is not a bug, and it turns out to be useful: because
zero-shot never trains, its score measures the *intrinsic difficulty* of each test set. All
three come out the same, so the three test sets are comparably hard — which is what licenses
attributing the fine-tuned model's spread across splits to training rather than to one split
being easier. It is a free control.

## 3. Is the machinery working at all?

Before concluding "the model can't do this", check that the scoring is doing what it should.
The direct test is wild-type recovery: mask an interface position and ask whether the model's
top prediction is the residue that is actually there. Chance is 1/20 = 5%.

In [5]:
@torch.no_grad()
def wt_recovered(sequence: str, position: int, wt_aa: str) -> bool:
    enc = tok(sequence, return_tensors="pt")
    ids = enc["input_ids"].clone()
    ids[0, position + 1] = tok.mask_token_id
    logits = model(input_ids=ids, attention_mask=enc["attention_mask"]).logits
    return tok.convert_ids_to_tokens(int(logits[0, position + 1].argmax())) == wt_aa

# Sample ACROSS complexes rather than from the smallest ones used above: short
# chains are unrepresentative, and a biased subsample would understate recovery.
# Seeded, so the number below is stable.
mid = complexes[complexes.sequence.str.len() <= 700]
check = (mutations[mutations.pdb_field.isin(mid.pdb_field)]
         .drop_duplicates(["pdb_field", "mutation_index"])
         .sample(200, random_state=0))
hits = [wt_recovered(seqs[r.pdb_field], r.mutation_index, r.wt_aa) for r in check.itertuples()]

print(f"wild-type recovery at {len(hits)} masked interface positions "
      f"across {check.pdb_field.nunique()} complexes: {np.mean(hits):.0%}")
print(f"chance is 5%. The full 35M run over all {len(mutations):,} rows gives 20%.")
print("\nWell above chance, so the machinery is correct. The model simply is not")
print("scoring the thing we need.")

wild-type recovery at 200 masked interface positions across 89 complexes: 16%
chance is 5%. The full 35M run over all 4,829 rows gives 20%.

Well above chance, so the machinery is correct. The model simply is not
scoring the thing we need.


## 4. The ablation that explains it

Here is the diagnostic worth the notebook. Score every mutation **twice**:

- `--context complex` — the mutated chain concatenated with its binding partner
- `--context chain` — the mutated chain **alone**, partner deleted entirely

If masked-marginal were reading anything about the *interface*, deleting the entire binding
partner should wreck it.

In [6]:
a = ROOT / "results/preds_zero_shot_esm2_t30_150M_UR50D.csv"
b = ROOT / "results/preds_zero_shot_esm2_t30_150M_UR50D_context-chain.csv"

if a.exists() and b.exists():
    both = (pd.read_csv(a).rename(columns={"y_pred": "complex"})
            .merge(pd.read_csv(b).rename(columns={"y_pred": "chain"}), on="uid"))
    rho = stats.spearmanr(both["complex"], both["chain"]).statistic
    print(f"n = {len(both):,} mutations scored both ways")
    print(f"agreement between the two scorings: Spearman ρ = {rho:.3f}")
else:
    print("prediction CSVs are gitignored (regenerable); recorded value: ρ = 0.888")
    print("run the two zero_shot commands above to rebuild them.")

n = 4,814 mutations scored both ways
agreement between the two scorings: Spearman ρ = 0.888


**ρ ≈ 0.89.** Deleting the entire binding partner barely changes the ranking.

That is the finding. ESM-2's masked-marginal is effectively **blind to the binding
partner** — it is scoring *"does this residue belong in this chain"*, which is what a
masked language model over single sequences is trained to know. It is not scoring *"does
this residue hold the interface together"*, which is what ΔΔG of binding asks.

This is why the floor is a floor, and it is the motivation for everything downstream:

- a model that sees **both** partners and is trained on ΔΔG labels (the siamese fine-tune);
- and, if that is not enough, an architecture with explicit **cross-chain attention** — which
  is what MINT adds, and it reports the strongest sequence-only SKEMPI result.

Also worth noting for scale: wild-type recovery is 25.0% scoring the full complex against
24.9% scoring the chain alone. The partner contributes essentially nothing even to the
model's *own* objective at these positions.

---

**What this baseline bought**

1. A floor, so any trained number can be judged against "no training at all".
2. A free control on test-set difficulty, since it never trains.
3. A mechanism, not just a number — the ablation says *why* it fails, which is what makes
   the next architectural step an argument rather than a guess.